# E2 — Análise do β: 5×2 + atan2 vs. 5×3

Lê uma pasta `results/e2/<config>/` (saída do `E2_treinamento_validacao`) e executa o **plano de análise
congelado** em `GOALS.md` → *GOALS 3 → Evaluation*. Não troque testes nem filtros olhando o resultado.

| sigla | β̂ | origem |
|---|---|---|
| **A** | atan2(v̂y, v̂x) | braço `5x2` |
| **B_out** | terceira saída evoluída | braço `5x3` |
| **B_atan2** | atan2(v̂y, v̂x) do próprio 5×3 | braço `5x3` |

Referência: β = atan2(col 17, col 18), em graus. Unidade de análise = (modelo, pista); os modelos de cada braço são
tratados como **amostras independentes** (as sementes não pareiam os braços).

| pergunta | comparação | teste |
|---|---|---|
| **Q1** o β melhora com a 3ª saída? | RMSE_β: B_out vs A · B_atan2 vs A | Mann–Whitney U |
| **Q2** a 3ª saída muda a qualidade de v_x, v_y? | RMSE_Vx e RMSE_Vy: 5×3 vs 5×2 | Mann–Whitney U |
| **Q3** β evoluído vs. pós-calculado, no mesmo modelo | RMSE_β: B_out vs B_atan2 | Wilcoxon (pareado) |

Convenção: toda diferença é **g1 − g2** com g1 = o braço/variante do 5×3 (negativo = o 5×3 erra menos). Alfa 0,05
bilateral, correção de Holm por família (Q1, Q2, Q3) sobre as 3 pistas de teste; Wang 2.1 (treino) e 2.2 (gate de
aprovação, logo enviesada) saem como exploratórias, sem correção. Todo teste traz também a diferença de medianas
com IC95 % bootstrap e o tamanho de efeito.

Só usa `predictions/*.npz`, `relatorio_validacao_rmse.csv`, `attempts.csv`, `attempts/*/equation.txt` e
`params.json` — **não precisa do `Database/`**. Tabelas agregadas e figuras vão para `<pasta>/analise/` (dentro de
`results/`, fora do git). A variável de ambiente `E2_RESULTS_DIR` sobrescreve a pasta (útil para rodar sem abrir o
Jupyter: `jupyter nbconvert --execute`). Se algum braço não estiver `DONE` com todos os aprovados, tudo sai carimbado
**PRELIMINAR**. Commite este notebook **sem saídas** (as figuras são séries derivadas dos dados da parceria).

## 1. Configuração

In [ ]:
import json
import os
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

# Tudo o que você normalmente mexe está nesta célula.
CONFIG      = "t48"
RESULTS_DIR = os.environ.get("E2_RESULTS_DIR")   # None = <repo>/results/e2/<CONFIG>
OUT_DIR     = None                               # None = <RESULTS_DIR>/analise
ARM_A, ARM_B = "5x2", "5x3"
TEST_TRACKS = ["testA", "testB", "testC"]        # evidência principal (plano congelado — não mudar)
ALPHA       = 0.05
N_BOOT      = 10_000
STATS_SEED  = 0
SHOW_FIG    = True                               # False = só salva os PNGs
ACEITAR     = {"5x3": 28}                        # braço fechado abaixo da meta por decisão registrada no GOALS.md

pd.options.display.float_format = "{:.4f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 30

ROLES = ["train", "val", "testA", "testB", "testC"]
PISTA = {"train": "Wang 2.1 (treino)", "val": "Wang 2.2 (gate)", "testA": "Wang 3.1 (teste A)",
         "testB": "Wang 8.1 (teste B)", "testC": "Wang 3.2 (teste C)"}
ARMS = [ARM_A, ARM_B]

In [ ]:
def _achar_raiz(inicio):
    for d in [inicio, *inicio.parents]:
        if (d / "src" / "mggp.py").exists():
            return d
    raise RuntimeError(f"não achei a raiz do repositório a partir de {inicio}")


RES = Path(RESULTS_DIR) if RESULTS_DIR else _achar_raiz(Path.cwd().resolve()) / "results" / "e2" / CONFIG
OUT = Path(OUT_DIR) if OUT_DIR else RES / "analise"
(OUT / "fig").mkdir(parents=True, exist_ok=True)

params = json.loads((RES / "params.json").read_text(encoding="utf-8"))
UNIT = params["beta_unit"]                      # unidade de β dentro dos .npz do braço 5x3
RUNS = params["runs"]
print(f"pasta: {RES}\nconfig {params['config']} · {RUNS} aprovados esperados por braço · β nos .npz em {UNIT}")

## 2. Estado dos braços e integridade

Antes de qualquer número: cada braço está `DONE`? Todo modelo aprovado do ledger tem `.npz` e linhas no relatório?
Sobrou arquivo de modelo sem registro no ledger (sintoma da falha em que o notebook de treino morre no meio)?

In [ ]:
def estado(arm):
    p = RES / arm / "STATUS.txt"
    return p.read_text(encoding="utf-8").splitlines()[0] if p.exists() else "SEM STATUS"


ledger = {a: pd.read_csv(RES / a / "attempts.csv") for a in ARMS}
rel = {a: pd.read_csv(RES / a / "relatorio_validacao_rmse.csv") for a in ARMS}

linhas, avisos = [], []
modelos = {}
for a in ARMS:
    led = ledger[a]
    aprov = led[led["status"] == "approved"]
    tem_npz = {m for m in aprov["model"] if (RES / a / "predictions" / f"{m}.npz").exists()}
    no_rel = set(rel[a]["Modelo"])
    modelos[a] = sorted(tem_npz & no_rel, key=lambda m: int(m.rsplit("_", 1)[1]))
    pkl = {p.stem for p in (RES / a / "models").glob("*.pkl")}
    sem_ledger = sorted(pkl - set(aprov["model"]))
    if set(aprov["model"]) - tem_npz:
        avisos.append(f"[{a}] aprovados no ledger sem .npz: {sorted(set(aprov['model']) - tem_npz)}")
    if set(aprov["model"]) - no_rel:
        avisos.append(f"[{a}] aprovados no ledger sem linhas no relatório: {sorted(set(aprov['model']) - no_rel)}")
    if sem_ledger:
        avisos.append(f"[{a}] modelos em models/ sem linha aprovada no ledger: {sem_ledger}")
    cont = led["status"].value_counts().to_dict()
    st = estado(a)
    if a in ACEITAR and st != "DONE" and len(modelos[a]) >= ACEITAR[a]:
        st = f"ACEITO com {ACEITAR[a]}"
    linhas.append({"braço": a, "STATUS": st, "aprovados (ledger)": len(aprov), "analisáveis": len(modelos[a]),
                   "esperados": ACEITAR.get(a, RUNS), "tentativas": cont})

display(pd.DataFrame(linhas).set_index("braço"))
PRELIMINAR = any(r["STATUS"] not in ("DONE", f"ACEITO com {r['esperados']}") or r["analisáveis"] < r["esperados"]
                 for r in linhas) or bool(avisos)
CARIMBO = "PRELIMINAR — não reportar" if PRELIMINAR else ""
for av in avisos:
    print("AVISO", av)
if PRELIMINAR:
    print("\n" + "!" * 90 + "\n!! PRELIMINAR: braço incompleto ou inconsistente — nada daqui entra em docs/results.md\n" + "!" * 90)
(OUT / "ESTADO.txt").write_text(
    ("PRELIMINAR\n" if PRELIMINAR else "FINAL\n") + "\n".join(f"{r['braço']}: {r['STATUS']} {r['analisáveis']}/{r['esperados']}"
                                                              for r in linhas) + "\n" + "\n".join(avisos) + "\n",
    encoding="utf-8")

## 3. Métricas recalculadas a partir dos `.npz`

Tudo é recalculado de `{pista}_yd` / `{pista}_yp`; depois confere contra o `relatorio_validacao_rmse.csv` (se não
bater, algo no pipeline mudou e a análise para). Pares modelo×pista com `ok == False` (Free-Run explodiu) entram como
NaN e são contados à parte.

In [ ]:
def beta_deg(vy, vx):
    return np.degrees(np.arctan2(vy, vx))


def dif_ang(a, b):                       # a - b em graus, no intervalo [-180, 180)
    return (a - b + 180.0) % 360.0 - 180.0


def rmse(x):
    return float(np.sqrt(np.mean(np.square(x))))


def graus(x):
    return np.degrees(x) if UNIT == "rad" else x


def metricas_modelo(arm, modelo):
    z = np.load(RES / arm / "predictions" / f"{modelo}.npz", allow_pickle=False)
    okmap = rel[arm][rel[arm]["Modelo"] == modelo].set_index("Papel")["ok"].astype(bool)
    n_out = 3 if arm == ARM_B else 2
    vel, beta, reg, dref = [], [], [], 0.0
    for role in ROLES:
        yd, yp = z[f"{role}_yd"], z[f"{role}_yp"]
        if yd.shape != yp.shape or yd.shape[1] != n_out:
            raise ValueError(f"{arm}/{modelo}/{role}: formas {yd.shape} {yp.shape}, esperado {n_out} saídas")
        base = {"arm": arm, "modelo": modelo, "role": role, "ok": bool(okmap[role])}
        variantes = ["A"] if arm == ARM_A else ["B_out", "B_atan2"]
        if not base["ok"]:
            vel.append({**base, "rmse_vx": np.nan, "rmse_vy": np.nan})
            beta += [{**base, "variante": v, "rmse": np.nan, "bias": np.nan, "mae": np.nan, "p95": np.nan}
                     for v in variantes]
            continue
        vxd, vyd = yd[:, 0], yd[:, 1]
        b_true = beta_deg(vyd, vxd)
        if n_out == 3:                   # a 3ª coluna de yd tem de ser o mesmo β de referência
            dref = max(dref, float(np.abs(graus(yd[:, 2]) - b_true).max()))
        vel.append({**base, "rmse_vx": rmse(vxd - yp[:, 0]), "rmse_vy": rmse(vyd - yp[:, 1])})
        preds = {"A": beta_deg(yp[:, 1], yp[:, 0])} if arm == ARM_A else \
            {"B_out": graus(yp[:, 2]), "B_atan2": beta_deg(yp[:, 1], yp[:, 0])}
        for v, bh in preds.items():
            e = dif_ang(bh, b_true)
            beta.append({**base, "variante": v, "rmse": rmse(e), "bias": float(e.mean()),
                         "mae": float(np.abs(e).mean()), "p95": float(np.percentile(np.abs(e), 95))})
            for eixo, sinal in (("abs_beta", np.abs(b_true)), ("vx", vxd)):
                q = np.digitize(sinal, np.quantile(sinal, [0.25, 0.5, 0.75]))      # 0..3
                for k in range(4):
                    m = q == k
                    if m.any():
                        reg.append({**base, "variante": v, "eixo": eixo, "quartil": k + 1, "rmse": rmse(e[m])})
    return vel, beta, reg, dref


vel, beta, reg, dref_max = [], [], [], 0.0
for a in ARMS:
    for m in modelos[a]:
        v_, b_, r_, d_ = metricas_modelo(a, m)
        vel += v_
        beta += b_
        reg += r_
        dref_max = max(dref_max, d_)
vel, beta, reg = pd.DataFrame(vel), pd.DataFrame(beta), pd.DataFrame(reg)
print(f"{vel['modelo'].nunique()} nomes de modelo · {len(vel)} pares modelo×pista · "
      f"máx |yd[:,2] − atan2(yd)| no 5×3 = {dref_max:.2e} deg")
assert dref_max < 1e-6, "a 3ª coluna de yd do 5×3 não é atan2(vy, vx): unidade/definição mudou?"

In [ ]:
# Conferência cruzada com o relatório do treino (só pares ok)
pares = {("A", ARM_A): "RMSE_beta_atan2_deg", ("B_atan2", ARM_B): "RMSE_beta_atan2_deg", ("B_out", ARM_B): "RMSE_beta_out_deg"}
dif = []
for a in ARMS:
    r = rel[a][rel[a]["ok"].astype(bool)].rename(columns={"Modelo": "modelo", "Papel": "role"})
    v = vel[(vel["arm"] == a) & vel["ok"]].merge(r[["modelo", "role", "RMSE_Vx", "RMSE_Vy"]], on=["modelo", "role"])
    dif += [(v["rmse_vx"] - v["RMSE_Vx"]).abs().max(), (v["rmse_vy"] - v["RMSE_Vy"]).abs().max()]
    for (var, arm_), col in pares.items():
        if arm_ != a:
            continue
        b = beta[(beta["arm"] == a) & (beta["variante"] == var) & beta["ok"]].merge(
            r[["modelo", "role", col]], on=["modelo", "role"])
        dif.append((b["rmse"] - b[col]).abs().max())
print(f"maior diferença recalculado × relatório: {max(dif):.2e}")
assert max(dif) < 1e-8, "as métricas recalculadas dos .npz não batem com o relatório"

tidy = pd.concat([
    beta.rename(columns={"variante": "grupo", "rmse": "valor"}).assign(metrica="RMSE_beta"),
    vel.melt(id_vars=["arm", "modelo", "role", "ok"], value_vars=["rmse_vx", "rmse_vy"],
             var_name="metrica", value_name="valor")
    .assign(grupo=lambda d: np.where(d["arm"] == ARM_A, "A", "B"),
            metrica=lambda d: d["metrica"].map({"rmse_vx": "RMSE_Vx", "rmse_vy": "RMSE_Vy"})),
])[["arm", "modelo", "role", "ok", "grupo", "metrica", "valor"]]

beta.to_csv(OUT / "por_modelo_beta.csv", index=False)
vel.to_csv(OUT / "por_modelo_vel.csv", index=False)

## 4. Divergências, linha de base e resumo por pista

Um modelo que explode no Free-Run numa pista é **resultado**, não ruído: a taxa por braço e pista vem antes das
medianas. As linhas de base dão escala ao RMSE_β: `β̂ = 0` e `β̂ = média de β na pista de treino`.

In [ ]:
div = (vel.assign(divergiu=~vel["ok"]).groupby(["arm", "role"])
       .agg(modelos=("modelo", "nunique"), divergiram=("divergiu", "sum")).reset_index())
div["taxa"] = div["divergiram"] / div["modelos"]
div["pista"] = div["role"].map(PISTA)
display(div.pivot(index="pista", columns="arm", values=["divergiram", "modelos"]).reindex(
    [PISTA[r] for r in ROLES]))
quais = vel[~vel["ok"]][["arm", "modelo", "role"]]
if len(quais):
    print("pares que divergiram:", [f"{r.arm}/{r.modelo}/{r.role}" for r in quais.itertuples()])
div.to_csv(OUT / "divergencias.csv", index=False)

# linha de base: a referência é a mesma para todos os modelos da config -> basta um .npz
z0 = np.load(RES / ARM_A / "predictions" / f"{modelos[ARM_A][0]}.npz", allow_pickle=False)
b_ref = {r: beta_deg(z0[f"{r}_yd"][:, 1], z0[f"{r}_yd"][:, 0]) for r in ROLES}
b_med_treino = float(b_ref["train"].mean())
base = pd.DataFrame([{"role": r, "pista": PISTA[r], "amostras": len(b_ref[r]), "beta_media": b_ref[r].mean(),
                      "beta_dp": b_ref[r].std(), "rmse_beta_zero": rmse(b_ref[r]),
                      "rmse_beta_media_treino": rmse(b_ref[r] - b_med_treino)} for r in ROLES])
display(base.set_index("role"))
base.to_csv(OUT / "baselines.csv", index=False)

In [ ]:
ordem_var = ["A", "B_out", "B_atan2"]
ok_beta = beta[beta["ok"]]


def q1(x):
    return x.quantile(0.25)


def q3(x):
    return x.quantile(0.75)


resumo_beta = (ok_beta.groupby(["role", "variante"])["rmse"]
               .agg(n="count", mediana="median", q1=q1, q3=q3, media="mean", dp="std").reset_index())
resumo_beta["metrica"] = "RMSE_beta (deg)"
ok_tidy = tidy[tidy["ok"] & (tidy["metrica"] != "RMSE_beta")]
resumo_vel = (ok_tidy.groupby(["role", "grupo", "metrica"])["valor"]
              .agg(n="count", mediana="median", q1=q1, q3=q3, media="mean", dp="std").reset_index()
              .rename(columns={"grupo": "variante"}))
resumo = pd.concat([resumo_beta, resumo_vel], ignore_index=True)
resumo["role"] = pd.Categorical(resumo["role"], ROLES)
resumo = resumo.sort_values(["metrica", "role", "variante"]).reset_index(drop=True)
display(resumo[resumo["metrica"] == "RMSE_beta (deg)"].drop(columns="metrica"))
display(resumo[resumo["metrica"] != "RMSE_beta (deg)"])
resumo.to_csv(OUT / "resumo_por_pista.csv", index=False)

vies = ok_beta.groupby(["role", "variante"])["bias"].agg(["median", "min", "max"]).rename(
    columns={"median": "vies_mediano_deg", "min": "vies_min", "max": "vies_max"})
print("\nviés de β̂ (graus, β̂ − β): mediana entre modelos")
display(vies)

## 5. Testes (plano congelado)

Mann–Whitney (independentes) para Q1/Q2, Wilcoxon (pareado por modelo) para Q3, Holm por família sobre as pistas
de teste; treino e gate saem como exploratórios (sem correção). `dif_mediana` = diferença das medianas g1 − g2
(independentes) ou mediana das diferenças por modelo (pareado), sempre com IC95 % bootstrap (percentil, 10 000
reamostragens); `efeito` = δ de Cliff (g1 > g2 positivo) nos independentes e fração de modelos com g1 < g2 nos
pareados.

In [ ]:
def holm(p):
    p = np.asarray(p, dtype=float)
    ordem = np.argsort(p)
    aj, corrente = np.empty(len(p)), 0.0
    for i, j in enumerate(ordem):
        corrente = max(corrente, (len(p) - i) * p[j])
        aj[j] = min(1.0, corrente)
    return aj


def rotulo_delta(d):
    d = abs(d)
    return "desprezível" if d < 0.147 else "pequeno" if d < 0.33 else "médio" if d < 0.474 else "grande"


def comparar(metrica, g1, g2, role, pareado, rng):
    sub = tidy[(tidy["metrica"] == metrica) & (tidy["role"] == role) & tidy["ok"] & tidy["valor"].notna()]
    s1 = sub[sub["grupo"] == g1].set_index("modelo")["valor"]
    s2 = sub[sub["grupo"] == g2].set_index("modelo")["valor"]
    if pareado:
        comuns = s1.index.intersection(s2.index)
        s1, s2 = s1[comuns], s2[comuns]
    x, y = s1.to_numpy(), s2.to_numpy()
    lin = {"metrica": metrica, "pista": role, "g1": g1, "g2": g2, "n1": len(x), "n2": len(y),
           "mediana_g1": np.median(x) if len(x) else np.nan, "mediana_g2": np.median(y) if len(y) else np.nan}
    lin["dif_mediana"] = lin["mediana_g1"] - lin["mediana_g2"]
    lin.update(ic95_inf=np.nan, ic95_sup=np.nan, efeito=np.nan, p=np.nan)
    if min(len(x), len(y)) < 5:
        return lin
    if pareado:
        d = x - y
        lin["dif_mediana"] = float(np.median(d))          # pareado: mediana das diferenças por modelo
        boot = np.median(d[rng.integers(0, len(d), (N_BOOT, len(d)))], axis=1)
        lin["efeito"] = float(np.mean(d < 0))
        try:
            lin["p"] = float(stats.wilcoxon(x, y).pvalue)
        except ValueError:                      # todas as diferenças nulas
            lin["p"] = 1.0
    else:
        boot = (np.median(x[rng.integers(0, len(x), (N_BOOT, len(x)))], axis=1)
                - np.median(y[rng.integers(0, len(y), (N_BOOT, len(y)))], axis=1))
        u = stats.mannwhitneyu(x, y, alternative="two-sided")
        lin["efeito"], lin["p"] = 2 * float(u.statistic) / (len(x) * len(y)) - 1, float(u.pvalue)
    lin["ic95_inf"], lin["ic95_sup"] = np.percentile(boot, [2.5, 97.5])
    return lin


COMPARACOES = [("Q1", "RMSE_beta", "B_out", "A", False), ("Q1", "RMSE_beta", "B_atan2", "A", False),
               ("Q2", "RMSE_Vx", "B", "A", False), ("Q2", "RMSE_Vy", "B", "A", False),
               ("Q3", "RMSE_beta", "B_out", "B_atan2", True)]
rng = np.random.default_rng(STATS_SEED)
testes = pd.DataFrame([{"familia": fam, **comparar(met, g1, g2, role, par, rng)}
                       for fam, met, g1, g2, par in COMPARACOES for role in ROLES])
testes["confirmatorio"] = testes["pista"].isin(TEST_TRACKS)
testes["p_holm"] = np.nan
for idx in testes[testes["confirmatorio"]].groupby("familia").groups.values():
    ok_p = testes.loc[idx, "p"].notna()
    testes.loc[ok_p[ok_p].index, "p_holm"] = holm(testes.loc[ok_p[ok_p].index, "p"])
testes["significativo"] = testes["p_holm"] < ALPHA
tipo = testes["g2"].isin(["B_atan2"])
testes["efeito_rotulo"] = [rotulo_delta(e) if (pd.notna(e) and not t) else "" for e, t in zip(testes["efeito"], tipo, strict=True)]
testes["pista"] = pd.Categorical(testes["pista"], ROLES)
testes = testes.sort_values(["familia", "metrica", "g1", "pista"]).reset_index(drop=True)
if CARIMBO:
    print("###", CARIMBO, "###  (p-valores abaixo servem só para depurar o notebook)")
display(testes)
testes.to_csv(OUT / "testes.csv", index=False)

## 6. Onde o erro mora: por regime

RMSE de β̂ por quartil de |β| e de v_x (quartis da própria pista), mediana entre modelos e pistas de teste. O atan2
das velocidades preditas fica mal condicionado com v_x baixo — se o erro do A/B_atan2 se concentra no 1º quartil de
v_x, é isso, e não o modelo.

In [ ]:
reg_t = reg[reg["role"].isin(TEST_TRACKS)]
reg_res = (reg_t.groupby(["eixo", "quartil", "variante"])["rmse"]
           .agg(n="count", mediana="median", q1=q1, q3=q3).reset_index())
display(reg_res.pivot(index=["eixo", "quartil"], columns="variante", values="mediana"))
reg_res.to_csv(OUT / "erro_por_regime.csv", index=False)

## 7. Figuras

Salvas em `<pasta>/analise/fig/` (não versionadas: são séries/curvas derivadas dos dados da parceria).

In [ ]:
CORES = {"A": "tab:blue", "B_out": "tab:purple", "B_atan2": "tab:green", "B": "tab:purple"}
ROT = {"A": "5×2·atan2", "B_out": "5×3·saída 3", "B_atan2": "5×3·atan2", "B": "5×3"}


def fechar(fig, nome):
    if CARIMBO:
        fig.suptitle(f"{fig.get_suptitle() + chr(10) if fig.get_suptitle() else ''}{CARIMBO}", color="red", y=1.04)
    fig.savefig(OUT / "fig" / nome, dpi=110, bbox_inches="tight")
    plt.show() if SHOW_FIG else plt.close(fig)


def caixas(ax, dados, rotulos, cores):
    bp = ax.boxplot(dados, tick_labels=rotulos, patch_artist=True, showfliers=True)
    for patch, c in zip(bp["boxes"], cores, strict=True):
        patch.set_facecolor(c)
        patch.set_alpha(0.55)
    ax.grid(True, ls=":", alpha=0.6)
    ax.tick_params(axis="x", rotation=30)


# 1) RMSE_beta por pista, com as linhas de base
fig, axes = plt.subplots(1, len(ROLES), figsize=(3.4 * len(ROLES), 4), squeeze=False)
for ax, role in zip(axes[0], ROLES, strict=True):
    dados = [ok_beta[(ok_beta["role"] == role) & (ok_beta["variante"] == v)]["rmse"].to_numpy() for v in ordem_var]
    caixas(ax, dados, [ROT[v] for v in ordem_var], [CORES[v] for v in ordem_var])
    b = base.set_index("role").loc[role]
    ax.axhline(b["rmse_beta_zero"], color="k", ls="--", lw=0.9, label="β̂ = 0")
    ax.axhline(b["rmse_beta_media_treino"], color="gray", ls=":", lw=0.9, label="β̂ = média do treino")
    ax.set_title(PISTA[role] + (" *" if role not in TEST_TRACKS else ""), fontsize=10)
axes[0, 0].set_ylabel("RMSE de β [deg]")
axes[0, 0].legend(fontsize=7)
fig.suptitle("RMSE de β por variante (* = exploratória: treino/gate)", y=1.03)
fechar(fig, "rmse_beta_por_pista.png")

# 2) RMSE de v_x e v_y: o 5x3 mudou a qualidade da velocidade?
fig, axes = plt.subplots(1, 2, figsize=(11, 4), squeeze=False)
for ax, met in zip(axes[0], ["RMSE_Vx", "RMSE_Vy"], strict=True):
    sub = tidy[(tidy["metrica"] == met) & tidy["ok"]]
    pos, dados, cores, rots = [], [], [], []
    for i, role in enumerate(ROLES):
        for j, g in enumerate(["A", "B"]):
            pos.append(i * 3 + j)
            dados.append(sub[(sub["role"] == role) & (sub["grupo"] == g)]["valor"].to_numpy())
            cores.append(CORES[g])
    bp = ax.boxplot(dados, positions=pos, patch_artist=True, widths=0.8)
    for patch, c in zip(bp["boxes"], cores, strict=True):
        patch.set_facecolor(c)
        patch.set_alpha(0.55)
    ax.set_xticks([i * 3 + 0.5 for i in range(len(ROLES))], [PISTA[r].split(" (")[0] for r in ROLES])
    ax.set_title(f"{met} — azul = 5×2, roxo = 5×3", fontsize=10)
    ax.grid(True, ls=":", alpha=0.6)
fig.suptitle("v_x e v_y: 5×2 vs 5×3", y=1.03)
fechar(fig, "rmse_vx_vy_5x2_vs_5x3.png")

# 3) erro por regime
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), squeeze=False)
for ax, (eixo, tit) in zip(axes[0], [("abs_beta", "quartil de |β| (1 = menor)"), ("vx", "quartil de v_x (1 = menor)")],
                           strict=True):
    for v in ordem_var:
        s = reg_res[(reg_res["eixo"] == eixo) & (reg_res["variante"] == v)].sort_values("quartil")
        ax.plot(s["quartil"], s["mediana"], "-o", color=CORES[v], label=ROT[v])
        ax.fill_between(s["quartil"], s["q1"], s["q3"], color=CORES[v], alpha=0.12)
    ax.set_xticks([1, 2, 3, 4])
    ax.set_xlabel(tit)
    ax.set_ylabel("RMSE de β [deg]")
    ax.grid(True, ls=":", alpha=0.6)
axes[0, 0].legend(fontsize=8)
fig.suptitle("RMSE de β por regime (mediana e IQR entre modelos × pistas de teste)", y=1.03)
fechar(fig, "erro_por_regime.png")

In [ ]:
# 4) sobreposição no tempo: pista de teste mais difícil para o A; modelo = o de RMSE mediano de cada braço
dificil = (ok_beta[(ok_beta["variante"] == "A") & ok_beta["role"].isin(TEST_TRACKS)]
           .groupby("role")["rmse"].median().idxmax())


def modelo_mediano(arm, variante, role):
    s = ok_beta[(ok_beta["arm"] == arm) & (ok_beta["variante"] == variante) & (ok_beta["role"] == role)]
    return s.iloc[(s["rmse"] - s["rmse"].median()).abs().argsort().iloc[0]]["modelo"]


mA, mB = modelo_mediano(ARM_A, "A", dificil), modelo_mediano(ARM_B, "B_out", dificil)
zA = np.load(RES / ARM_A / "predictions" / f"{mA}.npz", allow_pickle=False)
zB = np.load(RES / ARM_B / "predictions" / f"{mB}.npz", allow_pickle=False)
b_true = b_ref[dificil]
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(b_true, color="k", lw=0.8, label="β medido (atan2 col 17, 18)")
ax.plot(beta_deg(zA[f"{dificil}_yp"][:, 1], zA[f"{dificil}_yp"][:, 0]), color=CORES["A"], lw=0.7, ls="--",
        label=f"A · 5×2 atan2 ({mA})")
ax.plot(graus(zB[f"{dificil}_yp"][:, 2]), color=CORES["B_out"], lw=0.7, ls=":", label=f"B_out · 5×3 saída 3 ({mB})")
ax.set_xlabel("amostra")
ax.set_ylabel("β [deg]")
ax.legend(fontsize=8, loc="upper right")
ax.grid(True, ls=":", alpha=0.6)
fig.suptitle(f"{PISTA[dificil]} — pista de teste mais difícil para o A; modelos de RMSE_β mediano", y=1.02)
fechar(fig, "sobreposicao_beta.png")

## 8. Equações

Primeira passada estrutural (a classificação fina é `scripts/inspect_equations.py`): tamanho do modelo, se a saída
mantém o termo `y_k` puro (persistência: o coeficiente próximo de 1 significa "valor anterior + correção") e se as
saídas de v_x, v_y usam o β realimentado (`y3`) — pergunta central do 5×3.

In [ ]:
_TOKEN = re.compile(r"\b(?:add|subtraction|mul|sign|q\d+|[uy]\d+)\b")
_NUM = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")


def ler_equacao(path):
    cabeca, _, cauda = Path(path).read_text(encoding="utf-8").partition("theta:")
    partes = re.split(r"Output (\d+):", cabeca)
    termos = {int(k): [t.strip() for t in blk.strip().splitlines() if t.strip()]
              for k, blk in zip(partes[1::2], partes[2::2], strict=True)}
    thetas = [[float(x) for x in _NUM.findall(b)] for b in re.findall(r"array\(\[(.*?)\]\)", cauda, re.S)]
    return termos, thetas


eq, faltou = [], []
for a in ARMS:
    n_out = 3 if a == ARM_B else 2
    led = ledger[a].dropna(subset=["model"]).set_index("model")
    for m in modelos[a]:
        p = RES / a / "attempts" / f"attempt_{int(led.loc[m, 'attempt']):03d}" / "equation.txt"
        if not p.exists():
            faltou.append(f"{a}/{m}")
            continue
        termos, thetas = ler_equacao(p)
        for s, ts in termos.items():
            th = thetas[s - 1]
            if len(th) != len(ts):
                raise ValueError(f"{p}: {len(ts)} termos e {len(th)} coeficientes na saída {s}")
            eq.append({"arm": a, "modelo": m, "saida": s, "n_termos": len(ts) - 1,
                       "n_nos": sum(len(_TOKEN.findall(t)) for t in ts),
                       "ar_coef": th[ts.index(f"y{s}")] if f"y{s}" in ts else np.nan,
                       "termos_com_outras_saidas": sum(
                           any(re.search(rf"\by{j}\b", t) for j in range(1, n_out + 1) if j != s) for t in ts)})
if faltou:
    print("equation.txt ausente:", faltou)
eq = pd.DataFrame(eq)
resumo_eq = (eq.groupby(["arm", "saida"])
             .agg(modelos=("modelo", "nunique"), nos_mediano=("n_nos", "median"),
                  com_termo_y_puro=("ar_coef", lambda s: s.notna().mean()), ar_coef_mediano=("ar_coef", "median"),
                  termos_outras_saidas_med=("termos_com_outras_saidas", "median")).reset_index())
display(resumo_eq)
eq.to_csv(OUT / "equacoes.csv", index=False)

# custo-benefício: RMSE_β médio nas pistas de teste vs. tamanho total do modelo (nós, todas as saídas)
tam = eq.groupby(["arm", "modelo"])["n_nos"].sum().rename("nos_total").reset_index()
rm = (ok_beta[ok_beta["role"].isin(TEST_TRACKS) & (ok_beta["variante"].isin(["A", "B_out"]))]
      .groupby(["arm", "modelo", "variante"])["rmse"].mean().reset_index())
cb = rm.merge(tam, on=["arm", "modelo"])
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for v in ["A", "B_out"]:
    s = cb[cb["variante"] == v]
    ax.scatter(s["nos_total"], s["rmse"], color=CORES[v], alpha=0.7, label=ROT[v])
ax.set_xlabel("nós totais do modelo (todas as saídas)")
ax.set_ylabel("RMSE de β médio nas pistas de teste [deg]")
ax.grid(True, ls=":", alpha=0.6)
ax.legend(fontsize=8)
fig.suptitle("Custo-benefício (o 5×3 tem uma saída a mais: compare o formato, não o nível)", y=1.02, fontsize=10)
fechar(fig, "custo_beneficio.png")
cb.to_csv(OUT / "custo_beneficio.csv", index=False)

## 9. Fechamento

In [ ]:
print(("\n" + "!" * 90 + f"\n!! {CARIMBO.upper()}\n" + "!" * 90) if PRELIMINAR else "FINAL — braços DONE ou aceitos (ACEITAR) e consistentes")
print(f"\nsaídas em {OUT}:")
for p in sorted(OUT.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT)}  ({p.stat().st_size / 1024:.0f} KB)")